# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, exploration, and processing of the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset contains mass spectrometry-based quantification of proteins from extracellular vesicles of human mast cells, annotated according to the [Croissant](https://mlcommons.org/croissant/) schema.

### Dataset Source
The dataset Croissant schema URL is:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading

Load the dataset metadata with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes safely
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")

print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.date_published}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Let's review the record sets defined in the dataset, and enumerate fields available (all referenced by their `@id`).

In [ ]:
# List record sets by their @id
record_sets = list(dataset.record_sets)
print("Available record sets in dataset (referenced by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}")

# For demonstration, show available fields in each record set
for rs in record_sets:
    print(f"\nFields in record set {rs['@id']}:")
    for field in rs['fields']:
        print(f"    - @id: {field['@id']}, name: {field.get('name', 'N/A')}")
    if 'columns' in rs:
        print("  Columns:")
        for col in rs['columns']:
            print(f"    - @id: {col['@id']}, name: {col.get('name', 'N/A')}")

## 3. Data Extraction

Load records from each record set into pandas DataFrames. All selection is handled dynamically by `@id`.

> _Tip: choose the main record set for downstream analysis — typically containing primary measurements. Use the cells above to check the `@id`s._

In [ ]:
# Fetch all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for the current record set
    # Some record sets may be metadata/empty, handle errors gracefully
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set @id: {record_set_id}")
        if not df.empty:
            print(f"Columns: {df.columns.tolist()}")
            print(df.head(2))
    except Exception as e:
        print(f"Skipping record set {record_set_id} (Error: {str(e)})")

# Just select the first non-empty DataFrame for analysis in the next steps
main_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_record_set_id = k
        break
if main_record_set_id is None:
    raise ValueError("No non-empty record sets found!")
print(f"\nUsing main record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Let's:
- Filter records where a numeric field exceeds a given threshold,
- Normalize a numeric variable (z-score),
- Optionally group by a biological/class label.

> The field IDs and grouping variable are chosen from those printed in previous steps. **All references use their exact Croissant `@id`.**

In [ ]:
# Pick a numeric field for filtering/normalization
# Example: cr:field/Coverage (if exists), otherwise choose molecular weight

df = dataframes[main_record_set_id]

# Try to autodetect a likely numeric field by name
possible_numeric_fields = [col for col in df.columns if 'coverage' in col.lower() or 'count' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower() or 'weight' in col.lower()]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # Just default to first column if nothing matches
    numeric_field_id = df.columns[0]
    print("WARNING: Could not find a likely numeric field, using first column.")

print(f"Using '{numeric_field_id}' as the numeric field for EDA.")

# Ensure field is numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
numeric_threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
filtered_df = df[df[numeric_field_id] > numeric_threshold].copy()

print(f"Filtered records: {len(filtered_df)} with {numeric_field_id} > {numeric_threshold:.2f}")

filtered_df[f"{numeric_field_id}_zscore"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_zscore"]].head())

group_candidates = [col for col in df.columns if any(
    kw in col.lower() for kw in ['description', 'accession', 'class', 'group']
)]
if group_candidates:
    group_by_field = group_candidates[0]
    grouped = filtered_df.groupby(group_by_field)[numeric_field_id].mean()
    print(f"\nGrouped filtered records by {group_by_field}, mean {numeric_field_id}:")
    print(grouped.head())
else:
    group_by_field = None
    print("No suitable group field found.")

## 5. Visualization

Visualize the distribution of the numeric field and (if present) show groupwise differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

if group_by_field:
    plt.figure(figsize=(8,5))
    # Show side-by-side boxplots for top-5 groups
    top_groups = filtered_df[group_by_field].value_counts().index[:5]
    subdf = filtered_df[filtered_df[group_by_field].isin(top_groups)]
    sns.boxplot(data=subdf, x=group_by_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_by_field} (Top Groups)")
    plt.show()

## 6. Conclusion

- Successfully loaded and explored the FAIR^2 dataset package using `mlcroissant`.
- Performed basic filtering, normalization, and summaries using fields referenced by their Croissant `@id`s.
- Previewed main record set structure and visualized numeric variables, grouped if applicable.

**Next Steps:**
- Deeper biologically-motivated analysis (e.g. differential abundance, correlation with covariates)
- Export or clean for machine learning applications.

For more, see the [mlcroissant documentation](https://mlcommons.org/croissant/).